In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("food-w4zm1").project("recipe-ingredients-2")
version = project.version(3)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.15
    Uninstalling idna-3.15:
      Successfully uninstalled idna-3.15
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Recipe-Ingredients-2-3 in yolov8:: 100%|██████████| 31336/31336 [00:04<00:00, 6675.81it/s] 


In [2]:
print(dataset.location)

/content/Recipe-Ingredients-2-3


In [3]:
import yaml

with open(f"{dataset.location}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

print("Classes:")
print(data["names"])
print()

print("Num classes:")
print(len(data["names"]))

Classes:
['bay_leaf', 'bell_pepper', 'broccoli', 'cabbage', 'carrot', 'cauliflower', 'chicken', 'chickpeas', 'coriander', 'cucumber', 'egg', 'eggplant', 'fish', 'garlic', 'ginger', 'kumquat', 'lemon', 'long_pepper', 'mutton', 'okra', 'onion', 'pork', 'potato', 'pumpkin', 'radish', 'salt', 'shrimp', 'small_pepper', 'spring_onion', 'tofu', 'tomato', 'turmeric']

Num classes:
32


In [4]:
from collections import Counter
import os

counter = Counter()

label_dir = f"{dataset.location}/train/labels"

for file in os.listdir(label_dir):
    with open(os.path.join(label_dir, file)) as f:
        for line in f:
            cls = int(line.split()[0])
            counter[cls] += 1

for idx, name in enumerate(data["names"]):
    print(name, counter[idx])

bay_leaf 14
bell_pepper 2921
broccoli 85
cabbage 284
carrot 1302
cauliflower 953
chicken 7170
chickpeas 24
coriander 561
cucumber 495
egg 405
eggplant 166
fish 107
garlic 6039
ginger 5721
kumquat 19122
lemon 2435
long_pepper 755
mutton 178
okra 174
onion 16714
pork 8173
potato 4402
pumpkin 145
radish 110
salt 65
shrimp 99
small_pepper 764
spring_onion 1
tofu 364
tomato 11968
turmeric 65


In [5]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.4 MB/s eta 0:00:00


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Recipe-Ingredients-2-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, 

In [7]:
metrics = model.val()

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,011,888 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1256.3±487.2 MB/s, size: 43.0 KB)
val: Scanning /content/Recipe-Ingredients-2-3/valid/labels.cache... 794 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 195.9Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 12, len(boxes) = 10208. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 5.2it/s 9.7s
                   all        794      10208      0.903      0.915      0.941      0.699
           bell_pepper        111        113      0.902      0.976      0.976       0.72
               cabbage     

In [8]:
model.export(
    format="tflite"
)

Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 36, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.82', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 237ms
Prepared 4 packages in 1.54s
Installed 4 packages in 384ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 5.0s, saved as '/content/runs/detect/

'/content/runs/detect/train/weights/best_saved_model/best_float32.tflite'

In [9]:
model.export(
    format="tflite",
    int8=True
)

Ultralytics 8.4.57 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 36, 8400) (6.0 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=coco8.yaml'

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 3.2Kfiles/s 0.0s
Dataset download success ✅ (0.6s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1103.2±260.3 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 1.5Kit/s 0.0s
val: New cache created: /content/datasets/coco8/labels/val.cache
WARNING ⚠️ TensorFlow SavedModel

'/content/runs/detect/train/weights/best_saved_model/best_int8.tflite'

In [11]:
import os

print("FP32:",
      os.path.getsize("/content/runs/detect/train/weights/best_saved_model/best_float32.tflite")/1024/1024,
      "MB")

print("INT8:",
      os.path.getsize("/content/runs/detect/train/weights/best_saved_model/best_int8.tflite")/1024/1024,
      "MB")

FP32: 11.735300064086914 MB
INT8: 3.199251174926758 MB
